## ⚙️ **Libraries Import**

In [2]:
# Set seed for reproducibility
SEED = 42

# Import necessary libraries
import os

# Set environment variables before importing modules
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['MPLCONFIGDIR'] = os.getcwd() + '/configs/'

# Suppress warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=Warning)

# Import necessary modules
import logging
import random
import numpy as np

# Set seeds for random number generators in NumPy and Python
np.random.seed(SEED)
random.seed(SEED)

# Import PyTorch
import torch
torch.manual_seed(SEED)
from torch import nn
# from torchsummary import summary
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import TensorDataset, DataLoader
logs_dir = "tensorboard"
!pkill -f tensorboard
%load_ext tensorboard
!mkdir -p models

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device("cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

# Import other libraries
import copy
import shutil
from itertools import product
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plot display settings
sns.set(font_scale=1.4)
sns.set_style('white')
plt.rc('font', size=14)
%matplotlib inline

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard
PyTorch version: 2.6.0+cu124
Device: cuda


In [3]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from collections import defaultdict
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
for dirname, _, filenames in os.walk('/kaggle/input/the-pirate-pain-dataset'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/the-pirate-pain-dataset/sample_submission.csv
/kaggle/input/the-pirate-pain-dataset/pirate_pain_test.csv
/kaggle/input/the-pirate-pain-dataset/pirate_pain_train_labels.csv
/kaggle/input/the-pirate-pain-dataset/pirate_pain_train.csv


## ⏳ **Data Loading**

In [4]:
df_train = pd.read_csv("/kaggle/input/train-data-cleaned/dataset.csv")
df_public_test = pd.read_csv("/kaggle/input/preprocessed/dataset.csv")
df_labels = pd.read_csv("/kaggle/input/the-pirate-pain-dataset/pirate_pain_train_labels.csv")

## 🔄 **Data Preprocessing**

In [5]:
# Map int64, float64, string to float32
mapping = {
    "one+peg_leg": 1,
    "one+hook_hand": 1,
    "one+eye_patch": 1,
    "two": 2,
}


for col in df_train:
    if df_train[col].dtype == 'int64' and col not in ['sample_index', 'time']:
        df_train[col] = df_train[col].astype(np.float32)
    elif df_train[col].dtype == 'float64':
        df_train[col] = df_train[col].astype(np.float32)
    elif df_train[col].dtype == 'object':
        df_train[col] = df_train[col].replace(mapping)
        df_train[col] = df_train[col].astype(np.float32)

for col in df_public_test:
    if df_public_test[col].dtype == 'int64' and col not in ['sample_index', 'time']:
        df_public_test[col] = df_public_test[col].astype(np.float32)
    elif df_public_test[col].dtype == 'float64':
        df_public_test[col] = df_public_test[col].astype(np.float32)
    elif df_public_test[col].dtype == 'object':
        df_public_test[col] = df_public_test[col].replace(mapping)
        df_public_test[col] = df_public_test[col].astype(np.float32)

In [6]:
label_mapping = {label: idx for idx, label in enumerate(df_labels['label'].unique())}
df_labels['label'] = df_labels['label'].map(label_mapping)
unique_users = df_labels['sample_index'].unique()
class_counts = np.bincount(df_labels["label"])

class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum()  # normalize
class_weights

array([0.06426252, 0.34934199, 0.58639549])

In [7]:
df_labels.head()

,sample_index,label
0,0,0
1,1,0
2,2,1
3,3,0
4,4,0


In [9]:
df_train.head()

,sample_index,time,joint_00,joint_01,joint_02,joint_03,joint_04,joint_05,joint_06,joint_07,...,joint_20,joint_21,joint_22,joint_23,joint_24,joint_25,joint_26,joint_27,joint_28,joint_29
0,0,0,0.777507,0.738252,0.779512,0.804419,0.714916,0.736643,0.639301,0.733981,...,0.0,0.0,0.0,0.0,0.0,0.0,0.014214,0.011376,0.018978,0.020291
1,0,1,0.806256,0.765147,0.761153,0.838021,0.735684,0.729533,0.654604,0.760554,...,0.0,0.0,0.0,0.0,0.0,0.0,0.010748,0.000000,0.009473,0.010006
2,0,2,0.767592,0.721439,0.772834,0.777832,0.724497,0.734962,0.692340,0.787647,...,0.0,0.0,0.0,0.0,0.0,0.0,0.013097,0.006830,0.017065,0.016856
3,0,3,0.666220,0.810416,0.763971,0.785928,0.679928,0.722504,0.589127,0.793524,...,0.0,0.0,0.0,0.0,0.0,0.0,0.009505,0.006274,0.020264,0.017981
4,0,4,0.774297,0.773366,0.772162,0.767017,0.747710,0.743156,0.677764,0.725437,...,0.0,0.0,0.0,0.0,0.0,0.0,0.004216,0.002132,0.023389,0.018477


In [10]:
df_public_test.head()

,sample_index,time,joint_00,joint_01,joint_02,joint_03,joint_04,joint_05,joint_06,joint_07,...,joint_20,joint_21,joint_22,joint_23,joint_24,joint_25,joint_26,joint_27,joint_28,joint_29
0,0,0,0.842535,0.845934,0.573001,0.395143,0.302871,0.157435,0.977430,0.950154,...,0.000031,0.000003,0.000004,0.000003,0.000003,0.000068,0.019372,0.066324,0.022228,0.013576
1,0,1,0.898836,0.814810,0.629339,0.469420,0.247027,0.144169,1.005981,0.993922,...,0.000031,0.000003,0.000004,0.000004,0.000003,0.000029,0.069747,0.080417,0.023650,0.038793
2,0,2,0.957765,0.890690,0.608661,0.444819,0.295657,0.113608,0.989857,0.982370,...,0.000048,0.000006,0.000004,0.000009,0.000004,0.000008,0.054968,0.058811,0.027023,0.054202
3,0,3,0.832596,0.751427,0.605583,0.504279,0.236913,0.145222,1.001372,0.944723,...,0.000031,0.000005,0.000004,0.000003,0.000004,0.000015,0.048695,0.047128,0.016151,0.024983
4,0,4,0.805972,0.810004,0.565170,0.514796,0.269761,0.069992,1.023796,0.948609,...,0.000045,0.000006,0.000004,0.000003,0.000003,0.000008,0.019762,0.031116,0.015618,0.017931


### Building the sequences

In [8]:
# List of feature columns
feature_cols = [col for col in df_train.columns if col not in ["sample_index", "time"]]

In [9]:
def build_sequences(df, df_labels, window, stride):
    """
    Build sliding window sequences WITH padding for the Pirate Pain dataset.

    Parameters
    ----------
    df : DataFrame
        Feature data (contains sample_index, time, and feature columns)
    df_labels : DataFrame
        Label data (sample_index, label_idx)
    window : int
        Number of timesteps per sequence
    stride : int
        Step between windows (controls overlap)

    Returns
    -------
    X : np.array   shape = (num_sequences, window, num_features)
    y : np.array   shape = (num_sequences,)
    """

    assert window % stride == 0, "window must be divisible by stride"

    X = []
    y = []

    for sid in df["sample_index"].unique():

        # Extract joint-related features per sequence
        temp = df[df["sample_index"] == sid][feature_cols].values.astype("float32")

        # Get label
        label = df_labels.loc[df_labels["sample_index"] == sid, "label"].iloc[0]

        # --- Padding section ---
        remainder = len(temp) % window
        if remainder > 0:
            pad_len = window - remainder
            padding = np.zeros((pad_len, temp.shape[1]), dtype="float32")
            temp = np.concatenate((temp, padding), axis=0)

        # Build sliding windows
        idx = 0
        while idx + window <= len(temp):
            X.append(temp[idx:idx + window])
            y.append(label)
            idx += stride

    return np.array(X), np.array(y)


In [10]:
def make_loader(ds, batch_size, shuffle, drop_last):
    # Determine optimal number of worker processes for data loading
    cpu_cores = os.cpu_count() or 2
    num_workers = max(2, min(4, cpu_cores))

    # Create DataLoader with performance optimizations
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
        pin_memory=True,  # Faster GPU transfer
        pin_memory_device="cuda" if torch.cuda.is_available() else "",
        prefetch_factor=4,  # Load 4 batches ahead
    )

## 🛠️ **Model Building**

In [11]:
def recurrent_summary(model, input_size):
    """
    Custom summary function that emulates torchinfo's output while correctly
    counting parameters for RNN/GRU/LSTM layers.

    This function is designed for models whose direct children are
    nn.Linear, nn.RNN, nn.GRU, or nn.LSTM layers.

    Args:
        model (nn.Module): The model to analyze.
        input_size (tuple): Shape of the input tensor (e.g., (seq_len, features)).
    """

    # Dictionary to store output shapes captured by forward hooks
    output_shapes = {}
    # List to track hook handles for later removal
    hooks = []

    def get_hook(name):
        """Factory function to create a forward hook for a specific module."""
        def hook(module, input, output):
            # Handle RNN layer outputs (returns a tuple)
            if isinstance(output, tuple):
                # output[0]: all hidden states with shape (batch, seq_len, hidden*directions)
                shape1 = list(output[0].shape)
                shape1[0] = -1  # Replace batch dimension with -1

                # output[1]: final hidden state h_n (or tuple (h_n, c_n) for LSTM)
                if isinstance(output[1], tuple):  # LSTM case: (h_n, c_n)
                    shape2 = list(output[1][0].shape)  # Extract h_n only
                else:  # RNN/GRU case: h_n only
                    shape2 = list(output[1].shape)

                # Replace batch dimension (middle position) with -1
                shape2[1] = -1

                output_shapes[name] = f"[{shape1}, {shape2}]"

            # Handle standard layer outputs (e.g., Linear)
            else:
                shape = list(output.shape)
                shape[0] = -1  # Replace batch dimension with -1
                output_shapes[name] = f"{shape}"
        return hook

    # 1. Determine the device where model parameters reside
    try:
        device = next(model.parameters()).device
    except StopIteration:
        device = torch.device("cpu")  # Fallback for models without parameters

    # 2. Create a dummy input tensor with batch_size=1
    dummy_input = torch.randn(1, *input_size).to(device)

    # 3. Register forward hooks on target layers
    # Iterate through direct children of the model (e.g., self.rnn, self.classifier)
    for name, module in model.named_children():
        if isinstance(module, (nn.Linear, nn.RNN, nn.GRU, nn.LSTM)):
            # Register the hook and store its handle for cleanup
            hook_handle = module.register_forward_hook(get_hook(name))
            hooks.append(hook_handle)

    # 4. Execute a dummy forward pass in evaluation mode
    model.eval()
    with torch.no_grad():
        try:
            model(dummy_input)
        except Exception as e:
            print(f"Error during dummy forward pass: {e}")
            # Clean up hooks even if an error occurs
            for h in hooks:
                h.remove()
            return

    # 5. Remove all registered hooks
    for h in hooks:
        h.remove()

    # --- 6. Print the summary table ---

    print("-" * 79)
    # Column headers
    print(f"{'Layer (type)':<25} {'Output Shape':<28} {'Param #':<18}")
    print("=" * 79)

    total_params = 0
    total_trainable_params = 0

    # Iterate through modules again to collect and display parameter information
    for name, module in model.named_children():
        if name in output_shapes:
            # Count total and trainable parameters for this module
            module_params = sum(p.numel() for p in module.parameters())
            trainable_params = sum(p.numel() for p in module.parameters() if p.requires_grad)

            total_params += module_params
            total_trainable_params += trainable_params

            # Format strings for display
            layer_name = f"{name} ({type(module).__name__})"
            output_shape_str = str(output_shapes[name])
            params_str = f"{trainable_params:,}"

            print(f"{layer_name:<25} {output_shape_str:<28} {params_str:<15}")

    print("=" * 79)
    print(f"Total params: {total_params:,}")
    print(f"Trainable params: {total_trainable_params:,}")
    print(f"Non-trainable params: {total_params - total_trainable_params:,}")
    print("-" * 79)

In [12]:
class RecurrentClassifier(nn.Module):
    """
    Generic RNN classifier (RNN, LSTM, GRU).
    Uses the last hidden state for classification.
    """
    def __init__(
            self,
            input_size,
            hidden_size,
            num_layers,
            num_classes,
            rnn_type='GRU',        # 'RNN', 'LSTM', or 'GRU'
            bidirectional=False,
            dropout_rate=0.2
            ):
        super().__init__()

        self.rnn_type = rnn_type
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.bidirectional = bidirectional

        # Map string name to PyTorch RNN class
        rnn_map = {
            'RNN': nn.RNN,
            'LSTM': nn.LSTM,
            'GRU': nn.GRU
        }

        if rnn_type not in rnn_map:
            raise ValueError("rnn_type must be 'RNN', 'LSTM', or 'GRU'")

        rnn_module = rnn_map[rnn_type]

        # Dropout is only applied between layers (if num_layers > 1)
        dropout_val = dropout_rate if num_layers > 1 else 0

        # Create the recurrent layer
        self.rnn = rnn_module(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,       # Input shape: (batch, seq_len, features)
            bidirectional=bidirectional,
            dropout=dropout_val
        )

        # Calculate input size for the final classifier
        if self.bidirectional:
            classifier_input_size = hidden_size * 2 # Concat fwd + bwd
        else:
            classifier_input_size = hidden_size

        # Final classification layer
        self.classifier = nn.Linear(classifier_input_size, num_classes)

    def forward(self, x):
        """
        x shape: (batch_size, seq_length, input_size)
        """

        # rnn_out shape: (batch_size, seq_len, hidden_size * num_directions)
        rnn_out, hidden = self.rnn(x)

        # LSTM returns (h_n, c_n), we only need h_n
        if self.rnn_type == 'LSTM':
            hidden = hidden[0]

        # hidden shape: (num_layers * num_directions, batch_size, hidden_size)

        if self.bidirectional:
            # Reshape to (num_layers, 2, batch_size, hidden_size)
            hidden = hidden.view(self.num_layers, 2, -1, self.hidden_size)

            # Concat last fwd (hidden[-1, 0, ...]) and bwd (hidden[-1, 1, ...])
            # Final shape: (batch_size, hidden_size * 2)
            hidden_to_classify = torch.cat([hidden[-1, 0, :, :], hidden[-1, 1, :, :]], dim=1)
        else:
            # Take the last layer's hidden state
            # Final shape: (batch_size, hidden_size)
            hidden_to_classify = hidden[-1]

        # Get logits
        logits = self.classifier(hidden_to_classify)
        return logits

## 🧮 **Network and Training Hyperparameters**

## 🧠 **Model Training**

In [13]:
# Initialize best model tracking variables
best_model = None
best_performance = float('-inf')

In [14]:
def train_one_epoch(model, train_loader, criterion, optimizer, scaler, device, l1_lambda=0, l2_lambda=0):
    """
    Perform one complete training epoch through the entire training dataset.

    Args:
        model (nn.Module): The neural network model to train
        train_loader (DataLoader): PyTorch DataLoader containing training data batches
        criterion (nn.Module): Loss function (e.g., CrossEntropyLoss, MSELoss)
        optimizer (torch.optim): Optimization algorithm (e.g., Adam, SGD)
        scaler (GradScaler): PyTorch's gradient scaler for mixed precision training
        device (torch.device): Computing device ('cuda' for GPU, 'cpu' for CPU)
        l1_lambda (float): Lambda for L1 regularization
        l2_lambda (float): Lambda for L2 regularization

    Returns:
        tuple: (average_loss, f1 score) - Training loss and f1 score for this epoch
    """
    model.train()  # Set model to training mode

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    # Iterate through training batches
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        # Move data to device (GPU/CPU)
        inputs, targets = inputs.to(device), targets.to(device)

        # Clear gradients from previous step
        optimizer.zero_grad(set_to_none=True)

        # Forward pass with mixed precision (if CUDA available)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = model(inputs)
            loss = criterion(logits, targets)

            # Add L1 and L2 regularization
            l1_norm = sum(p.abs().sum() for p in model.parameters())
            l2_norm = sum(p.pow(2).sum() for p in model.parameters())
            loss = loss + l1_lambda * l1_norm + l2_lambda * l2_norm

        # Backward pass with gradient scaling
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Accumulate metrics
        running_loss += loss.item() * inputs.size(0)
        predictions = logits.argmax(dim=1)
        all_predictions.append(predictions.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

        # print(f'targets:{targets}')
        # print(f'predictions:{predictions}')

    # Calculate epoch metrics
    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_f1 = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average='weighted'
    )

    return epoch_loss, epoch_f1

In [15]:
def validate_one_epoch(model, val_loader, criterion, device):
    """
    Perform one complete validation epoch through the entire validation dataset.

    Args:
        model (nn.Module): The neural network model to evaluate (must be in eval mode)
        val_loader (DataLoader): PyTorch DataLoader containing validation data batches
        criterion (nn.Module): Loss function used to calculate validation loss
        device (torch.device): Computing device ('cuda' for GPU, 'cpu' for CPU)

    Returns:
        tuple: (average_loss, accuracy) - Validation loss and accuracy for this epoch

    Note:
        This function automatically sets the model to evaluation mode and disables
        gradient computation for efficiency during validation.
    """
    model.eval()  # Set model to evaluation mode

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    # Disable gradient computation for validation
    with torch.no_grad():
        for inputs, targets in val_loader:
            # Move data to device
            inputs, targets = inputs.to(device), targets.to(device)

            # Forward pass with mixed precision (if CUDA available)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits = model(inputs)
                loss = criterion(logits, targets)

            # Accumulate metrics
            running_loss += loss.item() * inputs.size(0)
            predictions = logits.argmax(dim=1)
            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(targets.cpu().numpy())

    # Calculate epoch metrics
    epoch_loss = running_loss / len(val_loader.dataset)
    epoch_accuracy = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average='weighted'
    )

    return epoch_loss, epoch_accuracy

In [16]:
def log_metrics_to_tensorboard(writer, epoch, train_loss, train_f1, val_loss, val_f1, model):
    """
    Log training metrics and model parameters to TensorBoard for visualization.

    Args:
        writer (SummaryWriter): TensorBoard SummaryWriter object for logging
        epoch (int): Current epoch number (used as x-axis in TensorBoard plots)
        train_loss (float): Training loss for this epoch
        train_f1 (float): Training f1 score for this epoch
        val_loss (float): Validation loss for this epoch
        val_f1 (float): Validation f1 score for this epoch
        model (nn.Module): The neural network model (for logging weights/gradients)

    Note:
        This function logs scalar metrics (loss/f1 score) and histograms of model
        parameters and gradients, which helps monitor training progress and detect
        issues like vanishing/exploding gradients.
    """
    # Log scalar metrics
    writer.add_scalar('Loss/Training', train_loss, epoch)
    writer.add_scalar('Loss/Validation', val_loss, epoch)
    writer.add_scalar('F1/Training', train_f1, epoch)
    writer.add_scalar('F1/Validation', val_f1, epoch)

    # Log model parameters and gradients
    for name, param in model.named_parameters():
        if param.requires_grad:
            # Check if the tensor is not empty before adding a histogram
            if param.numel() > 0:
                writer.add_histogram(f'{name}/weights', param.data, epoch)
            if param.grad is not None:
                # Check if the gradient tensor is not empty before adding a histogram
                if param.grad.numel() > 0:
                    if param.grad is not None and torch.isfinite(param.grad).all():
                        writer.add_histogram(f'{name}/gradients', param.grad.data, epoch)

In [17]:
def fit(model, train_loader, val_loader, epochs, criterion, optimizer, scaler, device,
        l1_lambda=0, l2_lambda=0, patience=0, evaluation_metric="val_f1", mode='max',
        restore_best_weights=True, writer=None, verbose=10, experiment_name=""):
    """
    Train the neural network model on the training data and validate on the validation data.

    Args:
        model (nn.Module): The neural network model to train
        train_loader (DataLoader): PyTorch DataLoader containing training data batches
        val_loader (DataLoader): PyTorch DataLoader containing validation data batches
        epochs (int): Number of training epochs
        criterion (nn.Module): Loss function (e.g., CrossEntropyLoss, MSELoss)
        optimizer (torch.optim): Optimization algorithm (e.g., Adam, SGD)
        scaler (GradScaler): PyTorch's gradient scaler for mixed precision training
        device (torch.device): Computing device ('cuda' for GPU, 'cpu' for CPU)
        l1_lambda (float): L1 regularization coefficient (default: 0)
        l2_lambda (float): L2 regularization coefficient (default: 0)
        patience (int): Number of epochs to wait for improvement before early stopping (default: 0)
        evaluation_metric (str): Metric to monitor for early stopping (default: "val_f1")
        mode (str): 'max' for maximizing the metric, 'min' for minimizing (default: 'max')
        restore_best_weights (bool): Whether to restore model weights from best epoch (default: True)
        writer (SummaryWriter, optional): TensorBoard SummaryWriter object for logging (default: None)
        verbose (int, optional): Frequency of printing training progress (default: 10)
        experiment_name (str, optional): Experiment name for saving models (default: "")

    Returns:
        tuple: (model, training_history) - Trained model and metrics history
    """

    # Initialize metrics tracking
    training_history = {
        'train_loss': [], 'val_loss': [],
        'train_f1': [], 'val_f1': []
    }

    # Configure early stopping if patience is set
    if patience > 0:
        patience_counter = 0
        best_metric = float('-inf') if mode == 'max' else float('inf')
        best_epoch = 0

    print(f"Training {epochs} epochs...")

    # Main training loop: iterate through epochs
    for epoch in range(1, epochs + 1):

        # Forward pass through training data, compute gradients, update weights
        train_loss, train_f1 = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, device, l1_lambda, l2_lambda
        )

        # Evaluate model on validation data without updating weights
        val_loss, val_f1 = validate_one_epoch(
            model, val_loader, criterion, device
        )

        # Store metrics for plotting and analysis
        training_history['train_loss'].append(train_loss)
        training_history['val_loss'].append(val_loss)
        training_history['train_f1'].append(train_f1)
        training_history['val_f1'].append(val_f1)

        # Write metrics to TensorBoard for visualization
        if writer is not None:
            log_metrics_to_tensorboard(
                writer, epoch, train_loss, train_f1, val_loss, val_f1, model
            )

        # Print progress every N epochs or on first epoch
        if verbose > 0:
            if epoch % verbose == 0 or epoch == 1:
                print(f"Epoch {epoch:3d}/{epochs} | "
                    f"Train: Loss={train_loss:.4f}, F1 Score={train_f1:.4f} | "
                    f"Val: Loss={val_loss:.4f}, F1 Score={val_f1:.4f}")

        # Early stopping logic: monitor metric and save best model
        if patience > 0:
            current_metric = training_history[evaluation_metric][-1]
            is_improvement = (current_metric > best_metric) if mode == 'max' else (current_metric < best_metric)

            if is_improvement:
                best_metric = current_metric
                best_epoch = epoch
                torch.save(model.state_dict(), "models/"+experiment_name+'_model.pt')
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping triggered after {epoch} epochs.")
                    break

    # Restore best model weights if early stopping was used
    if restore_best_weights and patience > 0:
        model.load_state_dict(torch.load("models/"+experiment_name+'_model.pt'))
        print(f"Best model restored from epoch {best_epoch} with {evaluation_metric} {best_metric:.4f}")

    # Save final model if no early stopping
    if patience == 0:
        torch.save(model.state_dict(), "models/"+experiment_name+'_model.pt')

    # Close TensorBoard writer
    if writer is not None:
        writer.close()

    return model, training_history

## Implentation of the Cross Validation

In [18]:
from torch.utils.data import TensorDataset, DataLoader

# Placeholder for Weight Initialization
def initialize_weights(model, weight_init):
    """
    Applies the specified weight initialization scheme to the model.
    """
    if weight_init == 'xavier':
        # Example using Xavier/Glorot uniform initialization
        for name, param in model.named_parameters():
            if 'weight' in name and len(param.shape) > 1:
                torch.nn.init.xavier_uniform_(param)
            elif 'bias' in name:
                torch.nn.init.zeros_(param)
    elif weight_init == 'kaiming':
        # Example using Kaiming/He initialization
        for name, param in model.named_parameters():
            if 'weight' in name and len(param.shape) > 1:
                torch.nn.init.kaiming_uniform_(param, nonlinearity='relu')
            elif 'bias' in name:
                torch.nn.init.zeros_(param)
    elif weight_init == 'orthogonal':
        # Example using Orthogonal initialization
        for name, param in model.named_parameters():
            if 'weight' in name and len(param.shape) > 1:
                try:
                    torch.nn.init.orthogonal_(param)
                except ValueError:
                    # Orthogonal init fails for 1D tensors (e.g., biases)
                    pass
            elif 'bias' in name:
                torch.nn.init.zeros_(param)
    else:
        # Default PyTorch initialization
        pass # Do nothing
    return model

In [19]:
def k_shuffle_split_cross_validation_round_rnn(df, epochs, criterion, device,
                            k, batch_size, hidden_layers, hidden_size, learning_rate, dropout_rate,
                            window_size, stride, rnn_type, bidirectional, weight_init='default', score_func='f1',
                            l1_lambda=0, l2_lambda=0, patience=0, evaluation_metric="val_f1", mode='max',
                            restore_best_weights=True, writer=None, verbose=10, seed=42, experiment_name=""):
    """
    Perform K-fold shuffle split cross-validation with user-based splitting for time series data.

    Args:
        df: DataFrame with columns ['user_id', 'activity', 'x_axis', 'y_axis', 'z_axis', 'id']
        ... (existing parameters)
        weight_init (str): Weight initialization scheme ('xavier', 'kaiming', 'orthogonal', 'default'). 
        score_func (str): The primary scoring metric used for optimization ('f1', 'accuracy', etc.).
        ... (existing parameters)

    Returns:
        fold_losses: Dict with validation losses for each split
        fold_metrics: Dict with validation scores (based on score_func) for each split
        best_scores: Dict with best score (based on score_func) for each split plus mean and std
    """

    # Initialise containers for results across all splits
    fold_losses = {}
    fold_metrics = {}
    best_scores = {}
    
    # Dynamically set the evaluation metric key expected by the 'fit' function
    # In our case a good way to predict is the f1!
    evaluation_metric_key = f"val_{score_func}"

    # Initialise model architecture
    model = RecurrentClassifier(
        input_size=len(feature_cols),
        hidden_size=hidden_size,
        num_layers=hidden_layers,
        num_classes=3,
        dropout_rate=dropout_rate,
        bidirectional=bidirectional,
        rnn_type=rnn_type
    ).to(device)     
    
    # --- 1. APPLY WEIGHT INITIALIZATION AND STORE STATE ---
    # Apply the chosen initialization scheme to the model
    model = initialize_weights(model, weight_init)         
    
    # Store initial weights to reset model for each split
    initial_state = copy.deepcopy(model.state_dict())
    # ----------------------------------------------------

    # Iterate through K random splits
    for split_idx in range(k):

        if verbose > 0:
            print(f"Split {split_idx+1}/{k}")

        train_users, val_users = train_test_split(
            df_labels['sample_index'],
            test_size=0.2,              # 20% validation
            stratify=df_labels['label'],
            random_state=SEED
        )

        # Split the dataset into training, validation, and test sets based on user IDs
        df_train = df[df['sample_index'].isin(train_users)].copy()
        df_val = df[df['sample_index'].isin(val_users)].copy()

        df_train.shape[1:]
        
        if verbose > 0:
            print(f"  Training set shape: {df_train.shape}")
            print(f"  Validation set shape: {df_val.shape}")

        # Guard against division by zero (already in your original code) --> spazzatura        range_ = train_max - train_min + 1e-8
      
        # Build sequences using the existing build_sequences function
        X_train, y_train = build_sequences(df_train, df_labels, window=window_size, stride=stride)
        X_val, y_val = build_sequences(df_val, df_labels, window=window_size, stride=stride)
       
        if verbose > 0:
            print(f"  Training sequences shape: {X_train.shape}")
            print(f"  Validation sequences shape: {X_val.shape}")
       
        # Create PyTorch datasets (assuming numpy to tensor conversion)
        train_ds = TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(y_train).long())
        val_ds   = TensorDataset(torch.from_numpy(X_val).float(), torch.from_numpy(y_val).long())
       
        # Create data loaders
        train_loader = make_loader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)
        val_loader   = make_loader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False)
       
        # Reset model to initial weights for fair comparison across splits
        model.load_state_dict(initial_state)

        # Define optimizer with L2 regularization
        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=l2_lambda)

        # Enable mixed precision training for GPU acceleration
        split_scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda')) # Corrected to torch.cuda.amp

        # Create directory for model checkpoints
        os.makedirs(f"models/{experiment_name}", exist_ok=True)

        # Train model on current split
        model, training_history = fit(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=epochs,
            criterion=criterion,
            optimizer=optimizer,
            scaler=split_scaler,
            device=device,
            writer=writer,
            patience=patience,
            verbose=verbose,
            l1_lambda=l1_lambda,
            # --- USE THE DYNAMIC METRIC KEY HERE ---
            evaluation_metric=evaluation_metric_key, # e.g., 'val_f1' or 'val_accuracy'
            mode=mode,
            restore_best_weights=restore_best_weights,
            experiment_name=experiment_name+"/split_"+str(split_idx)
        )

        # --- DYNAMICALLY STORE RESULTS ---
        # The key to retrieve the score is evaluation_metric_key
        fold_losses[f"split_{split_idx}"] = training_history['val_loss']
        fold_metrics[f"split_{split_idx}"] = training_history[evaluation_metric_key]
        best_scores[f"split_{split_idx}"] = max(training_history[evaluation_metric_key])

    # Compute mean and standard deviation of best scores across splits
    all_best_scores = [best_scores[k] for k in best_scores.keys() if k.startswith("split_")]
    best_scores["mean"] = np.mean(all_best_scores)
    best_scores["std"] = np.std(all_best_scores)

    if verbose > 0:
        print(f"Best mean {score_func.upper()} score: {best_scores['mean']:.4f}±{best_scores['std']:.4f}")

    return fold_losses, fold_metrics, best_scores

## Implementation of the Grid Search

In [20]:
def grid_search_cv_rnn(df, param_grid, fixed_params, cv_params, verbose=True):
    """
    Execute grid search with K-shuffle-split cross-validation for RNN models on time series data.

    Args:
        df: DataFrame with columns ['user_id', 'activity', 'x_axis', 'y_axis', 'z_axis', 'id'] --> all the important cols
        param_grid: Dict of parameters to test, e.g. {'batch_size': [16, 32], 'rnn_type': ['LSTM', 'GRU']}
        fixed_params: Dict of fixed hyperparameters (hidden_size, learning_rate, window_size, stride, etc.)
        cv_params: Dict of CV settings (epochs, k, patience, criterion, scaler, device, etc.)
        verbose: Print progress for each configuration

    Returns:
        results: Dict with scores for each configuration
        best_config: Dict with best hyperparameter combination
        best_score: Best mean F1 score achieved
    """
    # Generate all parameter combinations
    param_names = list(param_grid.keys())
    param_values = list(param_grid.values())
    combinations = list(product(*param_values))

    results = {}
    best_score = -np.inf
    best_config = None

    total = len(combinations)

    for idx, combo in enumerate(combinations, 1):
        # Create current configuration dict
        current_config = dict(zip(param_names, combo))
        config_str = "_".join([f"{k}_{v}" for k, v in current_config.items()])
        
        print(" try ", config_str)
        
        if verbose:
            print(f"\nConfiguration {idx}/{total}:")
            for param, value in current_config.items():
                print(f"  {param}: {value}")

        # Merge current config with fixed parameters
        run_params = {**fixed_params, **current_config}

        # Execute cross-validation
        _, _, fold_scores = k_shuffle_split_cross_validation_round_rnn(
            df=df,
            experiment_name=config_str,
            **run_params,
            **cv_params
        )

        # Store results
        results[config_str] = fold_scores

        # Track best configuration
        if fold_scores["mean"] > best_score:
            best_score = fold_scores["mean"]
            best_config = current_config.copy()
            if verbose:
                print("  NEW BEST SCORE!")

        if verbose:
            print(f"  F1 Score: {fold_scores['mean']:.4f}±{fold_scores['std']:.4f}")

    return results, best_config, best_score

In [21]:
def plot_top_configurations_rnn(results, k_splits, top_n=5, figsize=(14, 7)):
    """
    Visualise top N RNN configurations with boxplots of F1 scores across CV splits.

    Args:
        results: Dict of results from grid_search_cv_rnn
        k_splits: Number of CV splits used
        top_n: Number of top configurations to display
        figsize: Figure size tuple
    """
    # Sort by mean score
    config_scores = {name: data['mean'] for name, data in results.items()}
    sorted_configs = sorted(config_scores.items(), key=lambda x: x[1], reverse=True)

    # Select top N
    top_configs = sorted_configs[:min(top_n, len(sorted_configs))]

    # Prepare boxplot data
    boxplot_data = []
    labels = []

    # Define a dictionary for replacements, ordered to handle prefixes correctly
    replacements = {
        'batch_size_': 'BS=',
        'learning_rate_': '\nLR=',
        'hidden_layers_': '\nHL=',
        'hidden_size_': '\nHS=',
        'dropout_rate_': '\nDR=',
        'window_size_': '\nWS=',
        'stride_': '\nSTR=',
        'rnn_type_': '\nRNN=',
        'bidirectional_': '\nBIDIR=',
        'l1_lambda_': '\nL1=',
        'l2_lambda_': '\nL2='
    }

    # Replacements for separators
    separator_replacements = {
        '_learning_rate_': '\nLR=',
        '_hidden_layers_': '\nHL=',
        '_hidden_size_': '\nHS=',
        '_dropout_rate_': '\nDR=',
        '_window_size_': '\nWS=',
        '_stride_': '\nSTR=',
        '_rnn_type_': '\nRNN=',
        '_bidirectional_': '\nBIDIR=',
        '_l1_lambda_': '\nL1=',
        '_l2_lambda_': '\nL2=',
        '_': ''
    }

    for config_name, mean_score in top_configs:
        # Extract best score from each split (auto-detect number of splits)
        split_scores = []
        for i in range(k_splits):
            if f'split_{i}' in results[config_name]:
                split_scores.append(results[config_name][f'split_{i}'])
        boxplot_data.append(split_scores)

        # Verify we have the expected number of splits
        if len(split_scores) != k_splits:
            print(f"Warning: Config {config_name} has {len(split_scores)} splits, expected {k_splits}")

        # Create readable label using the replacements dictionary
        readable_label = config_name
        for old, new in replacements.items():
            readable_label = readable_label.replace(old, new)

        # Apply separator replacements
        for old, new in separator_replacements.items():
             readable_label = readable_label.replace(old, new)

        labels.append(f"{readable_label}\n(μ={mean_score:.3f})")

    # Create plot
    fig, ax = plt.subplots(figsize=figsize)
    bp = ax.boxplot(boxplot_data, labels=labels, patch_artist=True,
                    showmeans=True, meanline=True)

    # Styling
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)

    # Highlight best configuration
    ax.get_xticklabels()[0].set_fontweight('bold')

    ax.set_ylabel('F1 Score')
    ax.set_xlabel('Configuration')
    ax.set_title(f'Top {len(top_configs)} RNN Configurations - F1 Score Distribution Across {k_splits} Splits')
    ax.grid(alpha=0.3, axis='y')

    plt.xticks(rotation=0, ha='center')
    plt.tight_layout()
    plt.show()

In [ ]:
weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

param_grid = {
    'batch_size': [32, 64],
    'learning_rate': [0.01, 0.001],
    'hidden_size': [64, 128],
    'rnn_type': ['GRU', 'LSTM']
}

# 2. Define Fixed Model/Hyperparameters (CONSTANT across all runs)
fixed_params = {
    'hidden_layers': 2,
    'dropout_rate': 0.4,
    'window_size': 40,
    'stride': 10,
    'bidirectional': True,
    'weight_init': 'xavier',
    'score_func': 'f1',         
    'l1_lambda': 0.0,
    'l2_lambda': 1e-4
}

# 3. Define Cross-Validation Settings (CONSTANT across all runs)
cv_params = {
    'epochs': 50,
    'k': 5,
    'patience': 10,
    'device': torch.device(device),
    'criterion': nn.CrossEntropyLoss(weight=weights).to(device),
    'mode': 'max',
    'seed': 42
}

results, best_config, best_score, best_history = grid_search_cv_rnn(
    df=df_train,
    param_grid=param_grid,
    fixed_params=fixed_params,
    cv_params=cv_params,
    verbose=True
)


 try  batch_size_32_learning_rate_0.01_hidden_size_64_rnn_type_GRU

Configuration 1/16:
  batch_size: 32
  learning_rate: 0.01
  hidden_size: 64
  rnn_type: GRU
Split 1/5
  Training set shape: (84480, 32)
  Validation set shape: (21280, 32)
  Training sequences shape: (6864, 40, 30)
  Validation sequences shape: (1729, 40, 30)
Training 50 epochs...
Epoch   1/50 | Train: Loss=0.9612, F1 Score=0.6531 | Val: Loss=0.8811, F1 Score=0.7715
Epoch  10/50 | Train: Loss=0.3005, F1 Score=0.9061 | Val: Loss=0.9727, F1 Score=0.8418
Early stopping triggered after 13 epochs.
Best model restored from epoch 3 with val_f1 0.8477
Split 2/5
  Training set shape: (84480, 32)
  Validation set shape: (21280, 32)
  Training sequences shape: (6864, 40, 30)
  Validation sequences shape: (1729, 40, 30)
Training 50 epochs...
Epoch   1/50 | Train: Loss=0.9454, F1 Score=0.6675 | Val: Loss=0.7691, F1 Score=0.8114
Epoch  10/50 | Train: Loss=0.2824, F1 Score=0.9109 | Val: Loss=0.8636, F1 Score=0.8285
Epoch  20/50 | Tr

## LSTM

In [ ]:
# @title Plot Hitory
# Create a figure with two side-by-side subplots (two columns)
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(18, 5))

# Plot of training and validation loss on the first axis
ax1.plot(training_history['train_loss'], label='Training loss', alpha=0.3, color='#ff7f0e', linestyle='--')
ax1.plot(training_history['val_loss'], label='Validation loss', alpha=0.9, color='#ff7f0e')
ax1.set_title('Loss')
ax1.legend()
ax1.grid(alpha=0.3)

# Plot of training and validation accuracy on the second axis
ax2.plot(training_history['train_f1'], label='Training f1', alpha=0.3, color='#ff7f0e', linestyle='--')
ax2.plot(training_history['val_f1'], label='Validation f1', alpha=0.9, color='#ff7f0e')
ax2.set_title('F1 Score')
ax2.legend()
ax2.grid(alpha=0.3)

# Adjust the layout and display the plot
plt.tight_layout()
plt.subplots_adjust(right=0.85)
plt.show()

### Plot Confusion Matrix

In [ ]:
# @title Plot Confusion Matrix
# Collect predictions and ground truth labels
val_preds, val_targets = [], []
with torch.no_grad():  # Disable gradient computation for inference
    for xb, yb in val_loader:
        xb = xb.to(device)

        # Forward pass: get model predictions
        logits = rnn_model(xb)
        preds = logits.argmax(dim=1).cpu().numpy()

        # Store batch results
        val_preds.append(preds)
        val_targets.append(yb.numpy())

# Combine all batches into single arrays
val_preds = np.concatenate(val_preds)
val_targets = np.concatenate(val_targets)

# Calculate overall validation metrics
val_acc = accuracy_score(val_targets, val_preds)
val_prec = precision_score(val_targets, val_preds, average='weighted')
val_rec = recall_score(val_targets, val_preds, average='weighted')
val_f1 = f1_score(val_targets, val_preds, average='weighted')
print(f"Accuracy over the validation set: {val_acc:.4f}")
print(f"Precision over the validation set: {val_prec:.4f}")
print(f"Recall over the validation set: {val_rec:.4f}")
print(f"F1 score over the validation set: {val_f1:.4f}")

# Generate confusion matrix for detailed error analysis
cm = confusion_matrix(val_targets, val_preds)

# Create numeric labels for heatmap annotation
labels = np.array([f"{num}" for num in cm.flatten()]).reshape(cm.shape)

# Visualise confusion matrix
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=labels, fmt='',
            cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix — Validation Set')
plt.tight_layout()
plt.show()

## Make predictions on public test

In [ ]:
from datetime import datetime

# --- 1. Re-use your build_sequences function from Cell 20 ---
# (Make sure 'feature_cols' is defined as it was in Cell 19)
# (Make sure WINDOW_SIZE=40 and STRIDE=10 are defined as in Cell 18)

# This function is needed to process the test data
def build_test_sequences(df, window, stride):
    X_test_sequences = []
    sample_indices = []
    
    # Group by sample_index
    for sid in df["sample_index"].unique():
        temp = df[df["sample_index"] == sid][feature_cols].values.astype("float32")
        
        # --- Padding section (same as training) ---
        remainder = len(temp) % window
        if remainder > 0:
            pad_len = window - remainder
            padding = np.zeros((pad_len, temp.shape[1]), dtype="float32")
            temp = np.concatenate((temp, padding), axis=0)

        # --- Sliding window (same as training) ---
        idx = 0
        sequences_for_this_sample = []
        while idx + window <= len(temp):
            sequences_for_this_sample.append(temp[idx:idx + window])
            idx += stride
            
        X_test_sequences.append(np.array(sequences_for_this_sample))
        sample_indices.append(sid)

    return X_test_sequences, sample_indices

# --- 2. Build the test sequences ---
# This will result in a list of arrays, e.g., (113 users, 13 windows, 40 timesteps, 34 features)
test_sequences, test_sample_indices = build_test_sequences(df_public_test, WINDOW_SIZE, STRIDE)

# --- 3. Make predictions window-by-window and aggregate ---
best_model.eval()
final_predictions = []

with torch.no_grad():
    for user_sequences in test_sequences:
        # Convert this user's 13 sequences to a tensor
        user_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
        
        # Get logits for all 13 windows at once
        # Shape: (13, 3)
        logits = best_model(user_tensor)
        
        # Convert to probabilities using softmax
        # Shape: (13, 3)
        probabilities = torch.nn.functional.softmax(logits, dim=1)
        
        # Average the probabilities across all 13 windows
        # Shape: (3)
        mean_probs = torch.mean(probabilities, dim=0)
        
        # Get the final class prediction for this user
        pred_class = torch.argmax(mean_probs).item()
        final_predictions.append(pred_class)

# --- 4. Map back to original labels ---
inverse_label_mapping = {v: k for k, v in label_mapping.items()}
predicted_labels = [inverse_label_mapping[p] for p in final_predictions]

# --- 5. Save submission ---
submission_df = pd.DataFrame({
    "sample_index": test_sample_indices,
    "label": predicted_labels
})

submission_filename = f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_submission.csv"
submission_df.to_csv(submission_filename, index=False)
print(f"{submission_filename} created successfully!")
print(submission_df.head())